# 1. N-gram Modeling

Example : 
- "I love machine"
- "I love Python"
- "I enjoy coding"

### Bigram Counts

| Bigram          | Count |
|-----------------|-------|
| (I, love)       | 2     |
| (I, enjoy)      | 1     |
| (love, machine) | 1     |
| (love, Python)  | 1     |
| (enjoy, coding) | 1     |

### Context Counts (first word of bigram)

| Word (context) | Count |
|----------------|-------|
| I              | 3     |
| love           | 2     |
| enjoy          | 1     |

---

## Compute MLE Probabilities


$P(\text{love} \mid I) = \frac{2}{3} \approx 0.667$

$P(\text{enjoy} \mid I) = \frac{1}{3} \approx 0.333$

$P(\text{machine} \mid love) = \frac{1}{2} = 0.5$

$P(\text{Python} \mid love) = \frac{1}{2} = 0.5$
 

---

## Explanation

- To predict the next word after a given word, look at **all bigrams that start with that word**.  
- Divide the count of each bigram by the total count of the context word.  
- The word with the **highest probability** is the predicted next word.  

**Example:**  
- Context = `"I"` → candidates `"love"` (0.667) and `"enjoy"` (0.333) → predicte


In [137]:
text = 'artificial intelligence improves data analysis, artificial intelligence powers modern applicatons.'

1. Tokenize and preprocess the corpus (lowercase, punctuation as tokens)

In [138]:
from nltk.tokenize import word_tokenize

tokens = [t.lower() for t in word_tokenize(text)]

2. Build a bigram model and a traigam model

In [139]:
from nltk.util import ngrams
from collections import Counter

bigrams = list(ngrams(tokens, 2))
trigrams = list(ngrams(tokens, 3))

bigram_freq = Counter(bigrams)
trigram_freq = Counter(trigrams)

3 . Compute probabilities using Maximum Likelihood Estimation (MLE)

In [140]:
bigram_context_counts = Counter([bg[0] for bg in bigrams])
trigram_context_counts = Counter([tg[:2] for tg in trigrams])

bigram_mle = {bg: count / bigram_context_counts[bg[0]] for bg, count in bigram_freq.items()}
trigram_mle = {tg: count / trigram_context_counts[tg[:2]] for tg, count in trigram_freq.items()}

print("Bigram MLE probabilities:")
for k, v in list(bigram_mle.items()):
    print(k, ":", v)

print("\nTrigram MLE probabilities:")
for k, v in list(trigram_mle.items()):
    print(k, ":", v)

Bigram MLE probabilities:
('artificial', 'intelligence') : 1.0
('intelligence', 'improves') : 0.5
('improves', 'data') : 1.0
('data', 'analysis') : 1.0
('analysis', ',') : 1.0
(',', 'artificial') : 1.0
('intelligence', 'powers') : 0.5
('powers', 'modern') : 1.0
('modern', 'applicatons') : 1.0
('applicatons', '.') : 1.0

Trigram MLE probabilities:
('artificial', 'intelligence', 'improves') : 0.5
('intelligence', 'improves', 'data') : 1.0
('improves', 'data', 'analysis') : 1.0
('data', 'analysis', ',') : 1.0
('analysis', ',', 'artificial') : 1.0
(',', 'artificial', 'intelligence') : 1.0
('artificial', 'intelligence', 'powers') : 0.5
('intelligence', 'powers', 'modern') : 1.0
('powers', 'modern', 'applicatons') : 1.0
('modern', 'applicatons', '.') : 1.0


4. Predict the next word for the context: "artificial intelligence"

In [141]:
def predict_next_word_bigram(context_words, bigram_mle):
    
    candidates = {tg[1]: prob for tg, prob in bigram_mle.items() if tg[:1] == tuple(context_words)}
    next_word = max(candidates, key=candidates.get)
    
    return next_word

def predict_next_word_trigram(context_words, trigram_mle):
    
    candidates = {tg[2]: prob for tg, prob in trigram_mle.items() if tg[:2] == tuple(context_words)}
    next_word = max(candidates, key=candidates.get)
    
    return next_word

c1 = 'artificial'
c2 = 'intelligence'

next_word_bi = predict_next_word_bigram([c1], bigram_mle)
print(f"Next word after '{c1}':", next_word_bi)
next_word_tri = predict_next_word_trigram([c1, c2], trigram_mle)
print(f"Next word after '{c1} {c2}':", next_word_tri)

Next word after 'artificial': intelligence
Next word after 'artificial intelligence': improves


5. Compare predictions from bigram and trigram models

- For the bigram model, the probability of the common collocation ('artificial', 'intelligence') is 1.0, indicating that whenever "artificial" appears in the training data, it is always followed by "intelligence.

- The trigram model, which considers two preceding words, provides more nuanced probabilities. For example, ('artificial', 'intelligence', 'improves') has probability 0.5, and ('artificial', 'intelligence', 'powers') also has probability 0.5

# 2. Data Sparsity & Smoothing Techniques: How smoothing changes probability distribution and model behavior

1. Use the corpus: "students study machine learning. Students study Data Science."

In [142]:
text = 'students study machine learning. Students study Data Science.'

tokens = [t.lower() for t in word_tokenize(text)]

2. Build a bigram model without smoothing

In [143]:
bigram = list(ngrams(tokens, 2))

bigram_freq = Counter(bigram)
bigram_context_freq = Counter(bg[0] for bg in bigram)

for bg, count in bigram_freq.items():
    context_count = bigram_context_freq[bg[0]]
    mle = count / context_count
    print(f"Bigram: {bg}, Count: {count}, Context Count: {context_count}, MLE: {mle}")

Bigram: ('students', 'study'), Count: 2, Context Count: 2, MLE: 1.0
Bigram: ('study', 'machine'), Count: 1, Context Count: 2, MLE: 0.5
Bigram: ('machine', 'learning'), Count: 1, Context Count: 1, MLE: 1.0
Bigram: ('learning', '.'), Count: 1, Context Count: 1, MLE: 1.0
Bigram: ('.', 'students'), Count: 1, Context Count: 1, MLE: 1.0
Bigram: ('study', 'data'), Count: 1, Context Count: 2, MLE: 0.5
Bigram: ('data', 'science'), Count: 1, Context Count: 1, MLE: 1.0
Bigram: ('science', '.'), Count: 1, Context Count: 1, MLE: 1.0


3. Apply Laplace smoothing to the same model

In [144]:
vocab = set(tokens)
v = len(vocab)

for bg, count in bigram_freq.items():
    context_count = bigram_context_freq[bg[0]]
    laplace_prob = (count + 1) / (context_count + v)
    print(f"Bigram: {bg}, Laplace Probability: {laplace_prob:.3f}")

Bigram: ('students', 'study'), Laplace Probability: 0.333
Bigram: ('study', 'machine'), Laplace Probability: 0.222
Bigram: ('machine', 'learning'), Laplace Probability: 0.250
Bigram: ('learning', '.'), Laplace Probability: 0.250
Bigram: ('.', 'students'), Laplace Probability: 0.250
Bigram: ('study', 'data'), Laplace Probability: 0.222
Bigram: ('data', 'science'), Laplace Probability: 0.250
Bigram: ('science', '.'), Laplace Probability: 0.250


4. Use the test sentence: "students study ai."
5. Compute:
- Sentence probability without smoothing
- Sentence probability with smoothing

In [145]:
test = 'students study ai.'
tokens = word_tokenize(test.lower())
bigram_test = list(ngrams(tokens, 2))

print("MLE\n")
for bg in bigram_test:
    count = bigram_freq.get(bg, 0)
    context_count = bigram_context_freq.get(bg[0], 0)
    if context_count == 0:
        print(f"Bigram: {bg}, MLE: undefined (context unseen)")
        continue
    else:
        mle = count / context_count
        print(f"Bigram: {bg}, MLE: {mle}")
print("="*50)
print("Laplace Smoothing\n")
for bg in bigram_test:
    count = bigram_freq.get(bg, 0)
    context_count = bigram_context_freq.get(bg[0], 0)
    laplace_prob = (count + 1) / (context_count + v)
    print(f"Bigram: {bg}, Laplace Probability: {laplace_prob:.3f}")

MLE

Bigram: ('students', 'study'), MLE: 1.0
Bigram: ('study', 'ai'), MLE: 0.0
Bigram: ('ai', '.'), MLE: undefined (context unseen)
Laplace Smoothing

Bigram: ('students', 'study'), Laplace Probability: 0.333
Bigram: ('study', 'ai'), Laplace Probability: 0.111
Bigram: ('ai', '.'), Laplace Probability: 0.143


# 3. Evaluate language models using Perplexity

1. Choose one dataset:
- Brown corpus (NLTK), or Wikipedia

In [146]:
from nltk.corpus import brown

sentences = brown.sents()

2. Preprocess the text (tokenization, lowercase)

In [147]:
tokens = []
for sent in sentences:
    for t in sent:
        tokens.append(t.lower())

tokens[:10]

['the',
 'fulton',
 'county',
 'grand',
 'jury',
 'said',
 'friday',
 'an',
 'investigation',
 'of']

3. Split the dataset: 80% for training, 20% for testing

In [148]:
tokens_size = len(tokens)
train_tokens_size = int(tokens_size * 0.8)

train_tokens = tokens[:train_tokens_size]
test_tokens = tokens[train_tokens_size:]

print(f'Train length: {len(train_tokens)}, Test length: {len(test_tokens)}')

Train length: 928953, Test length: 232239


3. Train two bigram models:
- Without smoothing


In [149]:
from nltk.util import ngrams
from collections import Counter

In [150]:
bigrams = list(ngrams(train_tokens, 2))
bigrams_freq = Counter(bigrams)

In [151]:
bigram_cc = Counter([bg[0] for bg in bigrams])
bigram_cc['are']

4142

In [152]:
import pandas as pd

mle_list = []
for bg, count in bigrams_freq.items():
    context = bg[0]
    mle = count/bigram_cc[context]
    mle_list.append({'bg' : bg, 'mle' : mle})

df = pd.DataFrame(mle_list)
df = df.sort_values(by='mle', ascending=False)
df.head(10)

,bg,mle
372227,"(gentleness, and)",1.0
64359,"(jumper, on)",1.0
372261,"(remorse, about)",1.0
31,"(presentments, that)",1.0
30,"(term-end, presentments)",1.0
64329,"(imperfection, here)",1.0
261482,"(fanshawe, of)",1.0
261473,"(homewards, ;)",1.0
261470,"(thither, and)",1.0
261452,"(1598/9, .)",1.0


- With Laplace smoothing

In [153]:
lp_list = []
v = len(train_tokens)
for bg, count in bigrams_freq.items():
    context = bg[0]
    lp = (count+1)/(bigram_cc[context] + v)
    lp_list.append({'bg' : bg, 'lp' : lp})

df = pd.DataFrame(lp_list)
df = df.sort_values(by='lp', ascending=False)
df.head(10)

,bg,lp
42,"(of, the)",0.009049
86,"(in, the)",0.005417
24,"(., the)",0.005372
677,"(,, and)",0.005143
110,"(,, the)",0.003363
163,"(to, the)",0.003065
6305,"(;, ;)",0.002504
127,"('', .)",0.002120
410,"(on, the)",0.002063
121,"(and, the)",0.001998


5. Compute perplexity on the test set for both models

In [154]:
bigrams = list(ngrams(test_tokens, 2))

mle_list = []
for bg in bigrams:
    bg_count = bigrams_freq.get(bg, 0)
    bg_cc = bigram_cc.get(bg[0], 0)
    if bg_cc == 0:
        continue
    
    mle = bg_count/bg_cc
    mle_list.append({'bigram' : bg, 'mle' : mle})

df_mle = pd.DataFrame(mle_list)
df_mle = df_mle.sort_values(by='mle', ascending=False)
df_mle.head(10)

,bigram,mle
120273,"(incapable, of)",1.0
208263,"(subjected, to)",1.0
175279,"(faltered, .)",1.0
199103,"(groomed, to)",1.0
194500,"(coolness, of)",1.0
138724,"(miseries, of)",1.0
122838,"(devoid, of)",1.0
98630,"(reassuringly, ,)",1.0
200507,"(magnificence, of)",1.0
25369,"(eluding, the)",1.0


In [155]:
v = len(test_tokens)

lp_list = []
for bg in bigrams:
    bg_count = bigrams_freq.get(bg, 0)
    bg_cc = bigram_cc.get(bg[0], 0)
    lp = (bg_count+1)/(bg_cc+v)
    lp_list.append({'Bigram' : bg, 'lp' : lp})

df_lp = pd.DataFrame(lp_list)
df_lp = df_lp.sort_values(by='lp', ascending=False)
df_lp.head(10)

,Bigram,lp
72330,"(of, the)",0.032897
11605,"(of, the)",0.032897
139950,"(of, the)",0.032897
125637,"(of, the)",0.032897
157578,"(of, the)",0.032897
140068,"(of, the)",0.032897
42523,"(of, the)",0.032897
119516,"(of, the)",0.032897
220373,"(of, the)",0.032897
220169,"(of, the)",0.032897


In [156]:
mle_list[:4]

[{'bigram': ('whole', 'lifetime'), 'mle': 0.0},
 {'bigram': ('lifetime', 'before'), 'mle': 0.0},
 {'bigram': ('before', 'you'), 'mle': 0.02435723951285521},
 {'bigram': ('you', 'to'), 'mle': 0.02204531537048377}]

In [158]:
import numpy as np

def compute_perplexity_score(bigram_list: list, bg_kind: str):
    probs = [bg[bg_kind] for bg in bigram_list]
    probs = [p if p > 0 else 1e-12 for p in probs]
    v = len(probs)
    log_sum = np.sum(np.log(probs))
    perplexity = np.exp(-log_sum / v)
    return perplexity

print(f'Perplexity score for bigram without smoothing: {compute_perplexity_score(mle_list, "mle"):.3f}')
print(f'Perplexity score for bigram with smoothing: {compute_perplexity_score(lp_list, "lp"):.3f}')


Perplexity score for bigram without smoothing: 68026.493
Perplexity score for bigram with smoothing: 22400.505


6. Compare the perplexity scores

+ Perplexity Results for Bigram Model

    | Model | Perplexity |
    |-------|------------|
    | MLE (without smoothing) | 68,026.49 |
    | Laplace-smoothed (with add-one smoothing) | 22,400.51 |

+ Interpretation

    - **MLE without smoothing** produced an extremely high perplexity. This is because many bigrams in the test set were unseen during training, giving them zero probabilities and making the model highly "surprised" by the data.

    - **Laplace-smoothed bigram model** significantly reduced perplexity by assigning a small non-zero probability to unseen bigrams, making predictions more reasonable and stable.

    - **Conclusion:** Smoothing is essential for bigram models when evaluating on a test set that may contain unseen word pairs. Although perplexity is still high due to the large vocabulary and test set size, the smoothed model demonstrates clearly better predictive capability than the raw MLE model.
